In [1]:
import findspark
findspark.init()

from pyspark.conf import SparkConf
from pyspark.sql import SparkSession
import pyspark.sql.functions as F

conf = SparkConf().setAppName("585").setMaster("local[4]")
spark = SparkSession.builder.config(conf = conf).getOrCreate()
spark

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
25/08/23 06:01:59 WARN Utils: Your hostname, de24, resolves to a loopback address: 127.0.1.1; using 192.168.29.229 instead (on interface enp0s3)
25/08/23 06:01:59 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/08/23 06:02:03 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [ ]:
'''
Table: Insurance

+-------------+-------+
| Column Name | Type  |
+-------------+-------+
| pid         | int   |
| tiv_2015    | float |
| tiv_2016    | float |
| lat         | float |
| lon         | float |
+-------------+-------+
pid is the primary key (column with unique values) for this table.
Each row of this table contains information about one policy where:
pid is the policyholder's policy ID.
tiv_2015 is the total investment value in 2015 and 
tiv_2016 is the total investment value in 2016.
lat is the latitude of the policy holder's city. 
It's guaranteed that lat is not NULL.
lon is the longitude of the policy holder's city. 
It's guaranteed that lon is not NULL.
 

Write a solution to report the sum of all total investment values in 2016 tiv_2016, 
for all policyholders who:

have the same tiv_2015 value as one or more other policyholders, and
are not located in the same city as any other policyholder 
(i.e., the (lat, lon) attribute pairs must be unique).
Round tiv_2016 to two decimal places.

The result format is in the following example.

 

Example 1:

Input: 
Insurance table:
+-----+----------+----------+-----+-----+
| pid | tiv_2015 | tiv_2016 | lat | lon |
+-----+----------+----------+-----+-----+
| 1   | 10       | 5        | 10  | 10  |
| 2   | 20       | 20       | 20  | 20  |
| 3   | 10       | 30       | 20  | 20  |
| 4   | 10       | 40       | 40  | 40  |
+-----+----------+----------+-----+-----+
Output: 
+----------+
| tiv_2016 |
+----------+
| 45.00    |
+----------+
Explanation: 
The first record in the table, like the last record, meets both of the two criteria.
The tiv_2015 value 10 is the same as the third and fourth records, and its location is unique.

The second record does not meet any of the two criteria. 
Its tiv_2015 is not like any other policyholders and its location is the same as the third record, 
which makes the third record fail, too.
So, the result is the sum of tiv_2016 of the first and last record, which is 45.
'''

In [2]:
data = [
(1,10,5 ,10,10),
(2,20,20,20,20),
(3,10,30,20,20),
(4,10,40,40,40)
]
schema = ['pid','tiv_2015','tiv_2016','lat','lon']

In [3]:
df = spark.createDataFrame(data=data,schema=schema)
df.show()

+---+--------+--------+---+---+
|pid|tiv_2015|tiv_2016|lat|lon|
+---+--------+--------+---+---+
|  1|      10|       5| 10| 10|
|  2|      20|      20| 20| 20|
|  3|      10|      30| 20| 20|
|  4|      10|      40| 40| 40|
+---+--------+--------+---+---+



In [4]:
tiv_2015_df = df.groupBy(F.col("tiv_2015"))\
                .agg(F.count(F.col("*")).alias("counts"))\
                .where(F.col("counts") > 1)

lat_lon_df = df.groupBy(F.col("lat"),F.col("lon"))\
               .agg(F.count(F.col("*")).alias("counts"))\
               .where(F.col("counts") == 1)

df.alias("T").join( tiv_2015_df.alias("T1"),
                    F.col("T.tiv_2015") == F.col("T1.tiv_2015"),
                    'inner')\
             .join( lat_lon_df.alias("T2"),
                    (F.col("T.lat") == F.col("T2.lat")) & 
                    (F.col("T.lon") == F.col("T2.lon")),
                    'inner'
                  )\
             .select(F.round(F.sum(F.col("tiv_2016")),2).alias("tiv_2016"))\
             .show()

+--------+
|tiv_2016|
+--------+
|      45|
+--------+



## SQL Solution 

<pre>
WITH Tiv_2015 as (
    SELECT  tiv_2015, count(*) 
    FROM Insurance
    GROUP BY tiv_2015 
    HAVING count(*) > 1
),
lat_lon as (
    SELECT lat, lon, count(*) 
    FROM Insurance
    GROUP BY lat, lon
    HAVING count(*) = 1   
)
SELECT ROUND(SUM(tiv_2016),2) as tiv_2016
FROM Insurance 
WHERE  tiv_2015 IN (SELECT tiv_2015 FROM Tiv_2015) AND
       (lat,lon) IN (SELECT lat, lon FROM lat_lon)
</pre>